In [1]:
import sys

sys.path.append("..")
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from numbers import Number
from qiskit.synthesis import LieTrotter
import numpy as np
import scipy as scipy
import matplotlib.pyplot as plt

### State preparation with 1 ancilla and 2 excitations on a total of 35 qubits

In [2]:
ancilla = 27


def prep_psi_0(qc: QuantumCircuit):
    qc.cx(27, 10)
    qc.cx(27, 11)
    qc.cx(27, 16)
    return qc


def prep_psi_0_by_0(qc: QuantumCircuit):
    qc.cx(27, 10, ctrl_state="0")
    qc.cx(27, 11, ctrl_state="0")
    qc.cx(27, 16, ctrl_state="0")
    return qc

### Test if state prep works as expected

In [3]:
from qiskit_aer import StatevectorSimulator
from qiskit_aer.primitives import SamplerV2

#sv = StatevectorSimulator()
sampler=SamplerV2(options={"backend_options": {"method": "matrix_product_state"}})
qc = QuantumCircuit(28)
qc.h(27)
qc = prep_psi_0(qc)
qc.measure_all()

job = sampler.run([qc])
res = job.result()

res[0].data.meas.get_counts()

{'1000000000010000110000000000': 531, '0000000000000000000000000000': 493}

In [4]:
qc = QuantumCircuit(28)
qc.h(27)
qc = prep_psi_0(qc)
qc = prep_psi_0_by_0(qc)
qc.measure_all()
job = sampler.run([qc])
res = job.result()

res[0].data.meas.get_counts()

{'1000000000010000110000000000': 526, '0000000000010000110000000000': 498}

## Hardware experiments

In [6]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()
backend = service.backend('ibm_torino')

t_steps = 5
num_qubits = 27
circuits = np.ndarray([t_steps, t_steps], dtype=QuantumCircuit)
H_tilde = np.zeros([t_steps, t_steps], dtype=complex)
S_tilde = np.zeros([t_steps, t_steps], dtype=complex)
delta_t = np.pi / 40
synth = LieTrotter(reps=1)

qubit_subset = (list(range(25, 30))
        + [35, 36]
        + list(range(42, 46))
        + list(range(47, 51))
        + [54, 55, 56]
        + list(range(61, 70))
)


cm = backend.coupling_map
reduced_map = cm.reduce(qubit_subset)
edge_list = {(a, b) if a < b else (b, a) for a, b in list(reduced_map.get_edges())}
#ent_dec = [(rev_map[i], rev_map[j]) for i, j in edge_list]


def Heisenberg(J, n, ent_map):
    if isinstance(J, Number):
        J = np.ones(len(ent_map)) * J
    else:
        assert len(J) == len(ent_map)
    String = "I" * n
    H = SparsePauliOp("I" * n, 0)
    for j, c in zip(J, ent_map):
        c = np.sort(c)
        XX = String[: c[0]] + "X" + String[c[0] + 1 : c[1]] + "X" + String[c[1] + 1 :]
        H += SparsePauliOp(XX, j)
        YY = String[: c[0]] + "Y" + String[c[0] + 1 : c[1]] + "Y" + String[c[1] + 1 :]
        H += SparsePauliOp(YY, j)
        ZZ = String[: c[0]] + "Z" + String[c[0] + 1 : c[1]] + "Z" + String[c[1] + 1 :]
        H += SparsePauliOp(ZZ, j)
    return H.chop()

J = [-0.2377, -0.2237, -0.6851, -0.2228, -0.8255, -0.5598, -0.2913, -0.1127, -0.1677, -0.6457,  0.9298, -0.3136, -0.8957, -0.2698, -0.9679,  0.9013, -0.9109, -0.2552, -0.93  , -0.6401,  0.6877, -0.9631, -0.2518, -0.239 , -0.0098,  0.656 , -0.9661]

H = Heisenberg(J, num_qubits, edge_list)

# Generate approximate time evolution, rather than using PauliEvolutionGate
Rxyz_circ = QuantumCircuit(2)
Rxyz_circ.rxx(2 * delta_t, 0, 1)
Rxyz_circ.ryy(2 * delta_t, 0, 1)
Rxyz_circ.rzz(2 * delta_t, 0, 1)
Rxyz_instr = Rxyz_circ.to_instruction(label="RXX+YY+ZZ")
qc_temp = QuantumCircuit(num_qubits)
for edge in edge_list:
    qc_temp.append(Rxyz_instr, edge)
time_evol = QuantumCircuit(num_qubits).compose(qc_temp)

#time_evol=PauliEvolutionGate(H,time=delta_t,synthesis=synth)
real_obs_H = SparsePauliOp("X", 1) ^ H
imag_obs_H = SparsePauliOp("Y", 1) ^ H
real_obs_S = SparsePauliOp("X", 1) ^ SparsePauliOp("I" * num_qubits, 1)
imag_obs_S = SparsePauliOp("Y", 1) ^ SparsePauliOp("I" * num_qubits, 1)


for j in range(t_steps):
    for i in range(j + 1):
        m = i
        n = j - i
        qc = QuantumCircuit(28)
        qc.h(27)
        qc = prep_psi_0(qc)
        for _ in range(n):
            qc.append(time_evol, range(27))
        qc = prep_psi_0_by_0(qc)
        for _ in range(m):
            qc.append(time_evol, range(27))
        circuits[i][j] = qc.copy()

init_layout = qubit_subset + [46]  # ancilla at qubit 55 on torino
pm = generate_preset_pass_manager(
    backend=backend, optimization_level=3, initial_layout=init_layout
)

circs = []
for i in range(t_steps):
    circs.append([])
    for j in range(t_steps):
        if circuits[i][j] is not None:
            circs[i].append(circuits[i][j])
        else:
            circs[i].append(circuits[j][i])

circs = [pm.run(circ) for circ in circs]

print(
    f"Max circuit depth:{max([circs[i][j].depth(lambda x: len(x.qubits)==2) for i in range(t_steps) for j in range(t_steps)])}"
)

Max circuit depth:39


# Circuit execution

## Run as individual jobs to allow more shots

In [ ]:
from qiskit_ibm_runtime import EstimatorV2, Batch, EstimatorOptions

estimator = EstimatorV2(mode=backend)
estimator.options.default_shots = 800000  # Try going upto 1000000
estimator.options.resilience_level = 2
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XY4"
estimator.options.resilience.zne.noise_factors = (1, 1.3, 1.6)  # use fractional folding
estimator.options.resilience.zne.extrapolator = (
    "linear"  # exponential fit showed high std's, use this instead.
)
estimator.options.experimental = {"execution_path": "gen3-turbo"}

est_jobs = []
for i in range(t_steps):
    for j in range(t_steps):
        if circuits[i][j] is not None:
            circ = circs[i][j]
        else:
            circ = circs[j][i]
        est_jobs.append(
            estimator.run(
                [
                    (
                        circ,
                        [
                            real_obs_H.apply_layout(circ.layout),
                            imag_obs_H.apply_layout(circ.layout),
                            real_obs_S.apply_layout(circ.layout),
                            imag_obs_S.apply_layout(circ.layout),
                        ],
                    )
                ]
            )
        )

In [ ]:
job_results = [job.result() for job in est_jobs]
print([job for job in jobs if job.errored()]) # check if any of the jobs errored.

In [ ]:
t_steps = 8
num_qubits = 34
H_tilde = np.zeros([t_steps, t_steps], dtype=complex)
S_tilde = np.zeros([t_steps, t_steps], dtype=complex)
idx = 0
for i in range(t_steps):
    for j in range(t_steps):
        if circuits[i][j] is not None:
            H_tilde[i, j] = (
                job_results[idx][0].data.evs[0] + 1j * job_results[idx][0].data.evs[1]
            )
            S_tilde[i, j] = (
                job_results[idx][0].data.evs[2] + 1j * job_results[idx][0].data.evs[3]
            )
        else:
            H_tilde[i, j] = np.conj(H_tilde[j, i])
            S_tilde[i, j] = np.conj(S_tilde[j, i])
        idx += 1

In [ ]:
def truncation(threshold, S):
    eigvals, eigvecs = np.linalg.eig(S)
    idx = eigvals.argsort()[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    D = len(eigvals)
    truncated_eigvals = []
    truncated_eigvecs = []
    # make truncated matrix whose rows are eigenvectors of S with eigval above threshold
    for i in range(D):
        if eigvals[i] >= threshold:
            truncated_eigvals.append(eigvals[i])
            truncated_eigvecs.append(eigvecs[i])
    truncated_eigvals = np.array(truncated_eigvals)
    truncated_eigvecs = np.array(truncated_eigvecs).T
    print(len(truncated_eigvals))
    return truncated_eigvecs


def E_vs_D(H_tilde, S_tilde, threshold_slope):
    GSEs = []
    ground_states = []
    for d in range(1, len(S_tilde) + 1):
        epsilon = threshold_slope * d
        H_temp = H_tilde[0:d, 0:d]
        S_temp = S_tilde[0:d, 0:d]
        V_eps = truncation(epsilon, S_temp)
        # solve GEVP
        A = V_eps.conj().T @ H_temp @ V_eps
        B = V_eps.conj().T @ S_temp @ V_eps
        E, c = scipy.linalg.eig(a=A, b=B)
        idx = E.argsort()[::-1]
        eigvals = E[idx]
        eigvecs = c[:, idx]
        GSEs.append(eigvals[-1])
        ground_states.append(eigvecs[:, -1])
        #print(min(eigvals))

    return GSEs, ground_states, V_eps

## Run on simulator for comparison

In [8]:
circuits[2,2]

In [ ]:
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

backend = service.backend("ibm_torino")

transpiled_circuits = np.empty([t_steps, t_steps], dtype=object)

for i in range(t_steps):
    for j in range(i):
        if circuits[i][j] is not None:
            transpiled_circuits[i,j] = transpile(circuits[i, j], basis_gates=["cz", "id", "rz", "sx", "x"])
        else:
            transpiled_circuits[j,i] = transpiled_circuits[i,j]

H_tilde_sim = np.zeros([t_steps, t_steps], dtype=complex)
S_tilde_sim = np.zeros([t_steps, t_steps], dtype=complex)

simulator = EstimatorV2(options={"backend_options": {"method":"matrix_product_state"}})
for i in range(t_steps):
    for j in range(t_steps):
        if circuits[i][j] is not None:
            sim_job = simulator.run(
                [
                    (
                        circuits[i][j],
                        [real_obs_H, imag_obs_H, real_obs_S, imag_obs_S],
                    )
                ]
            )
            H_tilde_sim[i, j] = (
                sim_job.result()[0].data.evs[0] + 1j * sim_job.result()[0].data.evs[1]
            )
            S_tilde_sim[i, j] = (
                sim_job.result()[0].data.evs[2] + 1j * sim_job.result()[0].data.evs[3]
            )
        else:
            sim_job = simulator.run(
                [
                    (
                        circuits[j][i],
                        [real_obs_H, imag_obs_H, real_obs_S, imag_obs_S],
                    )
                ]
            )
            H_tilde_sim[i, j] = (
                sim_job.result()[0].data.evs[0] - 1j * sim_job.result()[0].data.evs[1]
            )
            S_tilde_sim[i, j] = (
                sim_job.result()[0].data.evs[2] - 1j * sim_job.result()[0].data.evs[3]
            )

AerError: 'unknown instruction: circuit-168'

In [ ]:
GSEs, _,_ = E_vs_D(H_tilde_sim, S_tilde_sim, 1e-11)
plt.plot(GSEs, label='MPS')  # Ground-state energies
plt.plot(17.81817798453781*np.ones(t_steps), label='Exact')
plt.title("Ground state energy estimation with 2 exc., 1 anc., 35 qubits MPS Simulation",fontsize=7)
plt.legend()
plt.xlabel("Krylov subspace dimension")
plt.ylabel("Ground state energy")

## Exact diagonalisation for comparison

In [ ]:
# Method from Krylov tutorial
import itertools as it
import scipy as sp
from qiskit.quantum_info import Pauli


def n_particle_gs(H_op, n_qubits, n_exc):
    """
    Find the ground state of the n particle(excitation) sector
    """
    H_x = []
    for p, coeff in H_op.to_list():
        H_x.append(set([i for i, v in enumerate(Pauli(p).x) if v]))

    H_z = []
    for p, coeff in H_op.to_list():
        H_z.append(set([i for i, v in enumerate(Pauli(p).z) if v]))

    H_c = H_op.coeffs

    n_exc = n_exc
    sub_dimn = int(sp.special.comb(n_qubits, n_exc))
    print("n_exc", n_exc, ", subspace dimension", sub_dimn)

    n_particle_H = np.zeros((sub_dimn, sub_dimn), dtype=complex)

    sparse_vecs = [
        set(vec) for vec in it.combinations(range(n_qubits), r=n_exc)
    ]  # list all of the possible sets of n_exc indices of 1s in n_exc-particle states
    for i, i_set in enumerate(sparse_vecs):
        for j, j_set in enumerate(sparse_vecs):
            if len(i_set.symmetric_difference(j_set)) <= 2:

                for p_x, p_z, coeff in zip(H_x, H_z, H_c):

                    if i_set.symmetric_difference(j_set) == p_x:
                        sgn = ((-1j) ** len(p_x.intersection(p_z))) * (
                            (-1) ** len(i_set.intersection(p_z))
                        )
                    else:
                        sgn = 0

                    n_particle_H[i, j] += sgn * coeff

    eige, eigv = np.linalg.eigh(n_particle_H)
    gs_en = eige[0]  # min(eige)
    gs_vec = eigv[:, 0]
    print("n_exc particle ground state energy: ", gs_en)
    return gs_en, gs_vec, n_particle_H


n_qubits = 34
edge_list = edge_list
H = Heisenberg(1, n_qubits, edge_list)
en, state, subspace_h = n_particle_gs(H, n_qubits, 2)
# print(np.linalg.norm(subspace_h,ord=2))